# Data Fruit

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("DateFruit_Dataset.csv")

X = df.drop("Class", axis=1)
y = df["Class"]

In [2]:
X.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,SkewRB,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,0.6019,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,0.4134,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,0.9183,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,1.8028,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,0.8865,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666


In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

le = LabelEncoder()
y = le.fit_transform(y)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

# ANN

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_tensor = torch.tensor(X_train_sc, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [6]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_laoder = DataLoader(test_dataset, batch_size=32)

In [7]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 7),
        )
    def forward(self, x):
        return self.model(x)

In [8]:
model = ANN().to(device)
critera = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Train ANN

In [9]:
epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).long()

        optimizer.zero_grad()
        outputs = model(xb)

        loss = critera(outputs, yb)
        loss.backward()

        optimizer.step()    #update param
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    print(f"Epoch = {epoch+1} / {epochs} => loss = {train_loss}")

Epoch = 1 / 100 => loss = 1.7002293804417485
Epoch = 2 / 100 => loss = 1.100872130497642
Epoch = 3 / 100 => loss = 0.7182881236076355
Epoch = 4 / 100 => loss = 0.5374309511288352
Epoch = 5 / 100 => loss = 0.4511017462481623
Epoch = 6 / 100 => loss = 0.39741051844928577
Epoch = 7 / 100 => loss = 0.3676365173381308
Epoch = 8 / 100 => loss = 0.329963906303696
Epoch = 9 / 100 => loss = 0.29957149015820544
Epoch = 10 / 100 => loss = 0.2836941746265992
Epoch = 11 / 100 => loss = 0.27431825256865955
Epoch = 12 / 100 => loss = 0.2635321306145709
Epoch = 13 / 100 => loss = 0.24760467461917712
Epoch = 14 / 100 => loss = 0.225600466132164
Epoch = 15 / 100 => loss = 0.22156917336194412
Epoch = 16 / 100 => loss = 0.21743147982203442
Epoch = 17 / 100 => loss = 0.21619463679583176
Epoch = 18 / 100 => loss = 0.2036289903132812
Epoch = 19 / 100 => loss = 0.19116193525817082
Epoch = 20 / 100 => loss = 0.18603978694781012
Epoch = 21 / 100 => loss = 0.18247297746331795
Epoch = 22 / 100 => loss = 0.1739448

# Evaluate

In [10]:
model.eval()
tt = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_laoder:
        xb = xb.to(device)
        yb = yb.to(device).long()

        output = model(xb)
        _, predicted = torch.max(output, 1)

        correct += (predicted == yb).sum().item()
        tt += yb.size(0)

print("Accuracy : ", correct / tt * 100)

Accuracy :  93.88888888888889
